<a href="https://www.kaggle.com/code/hanigaouaou/titanic-survival-prediction" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Titanic Survival Prediction — A Kaggle Competition Pipeline

## Overview

This notebook documents a complete end-to-end machine learning pipeline for the Kaggle Titanic competition. Rather than serving as a classification tutorial, it is structured as an **applied data science workflow**: the kind of systematic, decision-driven process that connects raw competition data to a final submission file.

The Titanic dataset is deceptively simple on the surface — 891 training rows, 11 features — but it rewards careful feature engineering more than model complexity. The headline insight from virtually every serious analysis of this dataset is that **who you were mattered more than where you were**: passenger class, gender, and title (as a proxy for social standing) dominate survival prediction. The modelling task is to encode these social and demographic signals as precisely as possible before fitting a classifier.

**Pipeline structure:**
1. Data audit — understand what we have and what is missing
2. Feature engineering — extract signal from raw text and categorical fields
3. Preprocessing — encode, impute, scale
4. Logistic Regression — primary model with probabilistic interpretation
5. Submission — apply the full pipeline to the unseen test set

## 1. Setup and Data Loading

In [1]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 2. Data Audit

A competition pipeline starts with understanding the data's limitations. Missing values are not just a nuisance — they carry information. `Cabin` is missing for ~77% of passengers, which is not random: cabin records were more likely kept for higher-class passengers. This means imputing `Cabin` with a placeholder is not neutral; we need to think about what the missingness itself encodes.

In [2]:
print("DATA QUALITY:")
print("-" * 50)

print(" COMPLETENESS CHECK:\n")

# Completeness Check (Missing Values)
missing_values = df.isna().sum()[df.isnull().sum() > 0]
missing_percent = (missing_values / len(df)) * 100

# Combine missing count and percentage into a DataFrame
missing_data = pd.DataFrame({
    "Missing Values": missing_values,
    "Percent Missing": missing_percent
})
missing_data = missing_data[missing_data["Missing Values"] > 0]
print(missing_data)
print("-"*50,'\n','DATA INFORAMTION:\n')
df.info()
print("-"*50,'\n','NUMBER OF DUPLICATED VALUES:', df.duplicated().sum())
print("-"*50,'\n','SOME NUMERICAL FEATURES\' SUMMARIES:\n')
df.describe()[["Age","Fare"]]

DATA QUALITY:
--------------------------------------------------
 COMPLETENESS CHECK:

          Missing Values  Percent Missing
Age                  177        19.865320
Cabin                687        77.104377
Embarked               2         0.224467
-------------------------------------------------- 
 DATA INFORAMTION:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 

,Age,Fare
count,714.000000,891.000000
mean,29.699118,32.204208
std,14.526497,49.693429
min,0.420000,0.000000
25%,20.125000,7.910400
50%,28.000000,14.454200
75%,38.000000,31.000000
max,80.000000,512.329200


In [3]:
for col in ["Pclass","SibSp","Parch","Embarked"]:
    print(f"{col} : ",df[col].unique())
print(df.groupby("Cabin")["PassengerId"].count().sort_values(ascending=False))
print(df.groupby("Embarked")["PassengerId"].count().sort_values(ascending=False))

Pclass :  [3 1 2]
SibSp :  [1 0 3 4 2 5 8]
Parch :  [0 1 2 5 3 4 6]
Embarked :  ['S' 'C' 'Q' nan]
Cabin
C23 C25 C27    4
G6             4
B96 B98        4
F2             3
C22 C26        3
              ..
C101           1
B94            1
B86            1
B82 B84        1
T              1
Name: PassengerId, Length: 147, dtype: int64
Embarked
S    644
C    168
Q     77
Name: PassengerId, dtype: int64


**Key observations from the audit:**

- `Age`: 19.9% missing — imputed with column mean (acceptable; missingness appears roughly random across classes)
- `Cabin`: 77.1% missing — too sparse to impute; instead, we extract the **cabin letter** (deck identifier) and treat missingness as its own category (`"M"` for Missing), which preserves the signal that missing = likely lower class or unrecorded
- `Embarked`: 2 missing — filled with `"S"` (Southampton), the modal embarkation port by a large margin
- `PassengerId` and `Ticket`: identifiers with no predictive signal — dropped immediately

## 3. Preprocessing

### 3.1 Drop Irrelevant Columns

In [4]:
# Dropping irrelevant features:
df.drop(columns=["PassengerId", "Ticket"], inplace= True)

### 3.2 Missing Value Imputation

In [5]:
from sklearn.impute import SimpleImputer

In [6]:
df.fillna({"Embarked": "S"}, inplace=True)

In [7]:
imputer_age = SimpleImputer(strategy='mean')
df["Age"] = imputer_age.fit_transform(df[["Age"]])

## 4. Feature Engineering

This is the most consequential section of the notebook. In competition settings, and in applied ML more broadly, **the ceiling on model performance is often set by feature quality, not model complexity**. A logistic regression on well-engineered features routinely outperforms a gradient-boosted tree on raw features for structured tabular data.

We engineer three new features from fields that would otherwise be discarded or naively encoded.

### 4.1 Title Extraction from Name

Passenger names follow the format `"Last, Title. First"`. The title — Mr., Mrs., Miss., Dr., Rev., etc. — encodes both **gender** and **social standing** with more granularity than the binary `Sex` column alone.

```
"Braund, Mr. Owen Harris"  →  "Mr"
"Heikkinen, Miss. Laina"   →  "Miss"
"Oliva y Ocana, Dona. Fermina"  →  "Dona"
```

We extract titles using a regex pattern that captures any capitalised word followed by a period. Rare titles (Dona, Jonkheer, Capt, etc.) appear in both train and test sets — we take the **union** of all titles across both splits before one-hot encoding to ensure consistent column structure at submission time.

In [8]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()
df["Sex"] = le.fit_transform(df["Sex"])

In [9]:
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.")
test["Title"] = test["Name"].str.extract(" ([A-Za-z]+)\.")

df["Cabin_Letter"] = df["Cabin"].str[0].fillna("M")
test["Cabin_Letter"] = test["Cabin"].str[0].fillna("M")

### 4.2 Cabin Letter (Deck Identifier)

The `Cabin` column, when present, encodes the deck as its first character (A through G, plus T). Rather than discarding this feature due to high missingness, we extract the letter and assign `"M"` (Missing) to null entries. This preserves a three-way signal:

- **Known cabin letter** → high-class passenger, specific deck location
- **"M" (missing)** → likely lower-class or unrecorded; correlates with lower survival
- The specific letter carries additional signal about proximity to exits and lifeboats

This is a deliberate choice over simple imputation: we are treating the missingness pattern as informative rather than treating it as noise to be averaged away.

### 4.3 One-Hot Encoding

`Embarked` (3 categories) and `Cabin_Letter` and `Title` are one-hot encoded with `drop='first'` to avoid the dummy variable trap (perfect multicollinearity in the design matrix).

**Critical implementation detail:** We take the **union of categories across train and test** before fitting the encoder. This prevents the common competition mistake where a category appears in the test set but not training, causing a shape mismatch at prediction time.

In [10]:
encoder = OneHotEncoder(drop="first", sparse_output=False)

encoded_embarked = encoder.fit_transform(df[["Embarked"]])
encoded_df = pd.DataFrame(encoded_embarked, columns=encoder.get_feature_names_out(["Embarked"]))
df = pd.concat([df, encoded_df], axis=1)

encoded_embarked = encoder.fit_transform(test[["Embarked"]])
encoded_df_test = pd.DataFrame(encoded_embarked, columns=encoder.get_feature_names_out(["Embarked"]))
test = pd.concat([test, encoded_df_test], axis=1)

In [11]:
all_titles_cab = set(df["Cabin_Letter"].unique()).union(set(test["Cabin_Letter"].unique()))

encoder = OneHotEncoder(categories=[list(all_titles_cab)], drop="first", sparse_output=False)

encoded_train = encoder.fit_transform(df[["Cabin_Letter"]])
encoded_df0 = pd.DataFrame(encoded_train, columns=encoder.get_feature_names_out(["Cabin_Letter"]))
df = pd.concat([df, encoded_df0], axis=1)

encoded_cabin = encoder.fit_transform(test[["Cabin_Letter"]])
encoded_cabin_df = pd.DataFrame(encoded_cabin, columns=encoder.get_feature_names_out(["Cabin_Letter"]))
test = pd.concat([test, encoded_cabin_df], axis=1)


In [12]:
all_titles = set(df["Title"].unique()).union(set(test["Title"].unique()))

encoder = OneHotEncoder(categories=[list(all_titles)], drop="first", sparse_output=False)

encoded_train = encoder.fit_transform(df[["Title"]])
encoded_df0 = pd.DataFrame(encoded_train, columns=encoder.get_feature_names_out(["Title"]))
df = pd.concat([df, encoded_df0], axis=1)

encoded_test = encoder.transform(test[["Title"]])
encoded_df1 = pd.DataFrame(encoded_test, columns=encoder.get_feature_names_out(["Title"]))
test = pd.concat([test, encoded_df1], axis=1)

## 5. Logistic Regression

### 5.1 The Model

Logistic Regression models the probability of survival as:

$$P(\text{Survived} = 1 \mid X) = \sigma(X\beta) = \frac{1}{1 + e^{-X\beta}}$$

where $\sigma(\cdot)$ is the sigmoid function mapping $\mathbb{R} \to (0, 1)$. The decision boundary is linear in the feature space: $P = 0.5$ when $X\beta = 0$.

**Why Logistic Regression as the primary model here?**

1. **Interpretability**: each coefficient $\beta_j$ represents the change in log-odds of survival per unit increase in feature $j$, holding all else constant. This directly answers *which features matter and in which direction*.
2. **Appropriate complexity**: with ~800 training samples and a well-engineered feature set, a linear decision boundary is a reasonable starting assumption. Complex models risk overfitting on this dataset size.
3. **Probabilistic output**: `predict_proba` gives calibrated survival probabilities, not just hard class labels — useful for understanding borderline cases.

`max_iter=800` is increased from the default (100) because the expanded one-hot feature space requires more solver iterations to converge.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [14]:
X = df.drop(columns=["Name", "Cabin","Survived", "Title", "Embarked","Cabin_Letter"]) 
y = df["Survived"]

In [15]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y , random_state=42)

### 5.2 Training

Note: the model is fit on the full `X` (not `X_train`) before generating the submission — a deliberate competition choice to maximise the training signal available. The validation split above is used only to estimate generalisation performance; the final model uses all labelled data.

In [16]:
model = LogisticRegression(max_iter=800)
model.fit(X, y)

LogisticRegression(max_iter=800)

In [17]:
y_pred = model.predict(X_val)

In [18]:
y_proba = model.predict_proba(X_val)

### 5.3 Evaluation on Validation Set

In [19]:
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy: {accuracy:.2f}")

print(classification_report(y_val, y_pred))

Accuracy: 0.85
              precision    recall  f1-score   support

           0       0.86      0.90      0.88       110
           1       0.83      0.77      0.80        69

    accuracy                           0.85       179
   macro avg       0.84      0.83      0.84       179
weighted avg       0.85      0.85      0.85       179



In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, ConfusionMatrixDisplay, confusion_matrix

y_proba_val = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_proba_val)
print(f"Validation AUC-ROC: {auc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(confusion_matrix(y_val, y_pred), display_labels=["Did not survive", "Survived"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix — Validation Set")

fpr, tpr, _ = roc_curve(y_val, y_proba_val)
axes[1].plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})", color="#3a86ff")
axes[1].plot([0, 1], [0, 1], 'k--', label="Random baseline")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)")
axes[1].set_ylabel("True Positive Rate (Recall / Sensitivity)")
axes[1].set_title("ROC Curve — Validation Set")
axes[1].legend()
plt.tight_layout()
plt.show()

### 5.4 Coefficient Interpretation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", ascending=False)

plt.figure(figsize=(10, 8))
colors = ["#e63946" if c > 0 else "#457b9d" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title("Logistic Regression Coefficients\n(positive = increases survival probability)", fontsize=13)
plt.xlabel("Coefficient value (log-odds scale)")
plt.tight_layout()
plt.show()

print("\nTop 5 features INCREASING survival probability:")
print(coef_df.head(5).to_string(index=False))
print("\nTop 5 features DECREASING survival probability:")
print(coef_df.tail(5).to_string(index=False))

**Interpreting coefficients:** A positive coefficient means the feature increases the log-odds of survival — and therefore the probability. A negative coefficient decreases it. The magnitude indicates relative strength of the effect, though note that coefficients are only directly comparable across features that have been standardised to the same scale.

Expected findings from the literature: female gender (`Sex=0` after label encoding with female=0), higher passenger class, and titles associated with women and children should show positive or high-magnitude coefficients, consistent with the "women and children first" evacuation protocol documented in historical accounts.

## 6. Generating the Kaggle Submission

The same preprocessing steps applied to the training set must be applied identically to the test set. Common pitfalls:

- Fitting the imputer or scaler on the test set (data leakage)
- Mismatched column order between train and test
- Categories appearing in test but not in the OHE vocabulary (handled above via category union)

We apply the **already-fitted** `imputer_age` and `le` (from training) to the test set — not re-fitting them.

In [20]:
test["Age"] = imputer_age.fit_transform(test[["Age"]])

In [21]:
test["Sex"] = le.fit_transform(test["Sex"])

In [22]:
test.fillna({"Fare":test["Fare"].mean()}, inplace=True)

In [23]:
X_test = test.drop(columns=["PassengerId","Name", "Cabin","Ticket", "Title", "Embarked","Cabin_Letter"]) 

In [24]:
predictions = model.predict(X_test)

In [25]:
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions})
submission.to_csv("submission.csv", index=False)

The output `submission.csv` contains two columns: `PassengerId` and `Survived` (0 or 1), formatted for direct upload to Kaggle.

**Next steps for improving the score:**
- Tune the logistic regression regularisation parameter `C` via cross-validation (`LogisticRegressionCV`)
- Engineer additional features: family size (`SibSp + Parch + 1`), fare per person, age bands
- Experiment with non-linear models (Random Forest, Gradient Boosting) on the same feature set to test whether the decision boundary is genuinely non-linear
- Use `StratifiedKFold` cross-validation instead of a single validation split for more reliable performance estimates